# Procesamiento

#### Libraries

Currently Using

- Pandas
- Numpy
- Os


In [ ]:
import os
import traceback
import pandas as pd
from pandasgui import show
from IPython.display import display

pd.set_option("display.max_columns", None)

### Numero de archivos y Nombres de los deportes

In [ ]:
target_folder = "Datos_disciplinas"


def count_number(target_folder):
    try:
        file_count = sum(1 for entry in os.scandir(target_folder) if entry.is_file())
        print(f"Total de documentos: {file_count}")

    except FileNotFoundError:
        print(f"Error: no se encontró la carpeta '{target_folder}'.")


def ramas_d():
    sports = []
    files = []

    print("Lista de deportes:")

    for file in os.listdir(target_folder):
        if file.endswith(".xlsx"):
            files.append(file)

            name = file.rsplit(".", 1)[0]
            sports.append(name)

            print(f"{len(sports)}: {name}")

    return sports, files

### Aux


In [ ]:
def preguntar_sn(mensaje):
    while True:
        r = input(mensaje).strip().lower()
        if r in ("s", "si", "sí"):
            return True
        if r in ("n", "no"):
            return False
        print("Introduce s o n.")


def pedir_nombres(validos):
    while True:
        texto = input("Nombres exactos separados por comas (Enter para cancelar): ")
        nombres = [n.strip() for n in texto.split(",") if n.strip()]
        if not nombres:
            return None
        no_existen = [n for n in nombres if n not in validos]
        if not no_existen:
            return nombres
        print("No están en la lista:", ", ".join(no_existen))


def pedir_saltos(validos):
    while True:
        texto = input(
            "Números de los saltos a eliminar, separados por comas "
            "(Enter para cancelar): "
        )
        partes = [p.strip() for p in texto.split(",") if p.strip()]
        if not partes:
            return None
        try:
            numeros = [int(p) for p in partes]
        except ValueError:
            print("Solo se permiten números enteros.")
            continue
        invalidos = [n for n in numeros if n not in validos]
        if not invalidos:
            return numeros
        print("No pertenecen a los atletas seleccionados:", invalidos)




### Jump Count


In [ ]:
def jump_count(df, sheet_name):

    df_tmp = df.copy()

    # El índice identifica cada salto, así que debe ser único
    if not df_tmp.index.is_unique:
        df_tmp = df_tmp.reset_index(drop=True)

    columnas_base = [
        "Athlete",
        "Test Type",
        "Trial",
        "Jump Height (Imp-Mom) [cm]",
        "Concentric Impulse (Left) [N s]",
        "Concentric Impulse (Right) [N s]",
        "Concentric RFD (Left) [N/s]",
        "Concentric RFD (Right) [N/s]",
        "Eccentric Braking RFD (Left) [N/s]",
        "Eccentric Braking RFD (Right) [N/s]",
        "RSI-modified (Imp-Mom) [m/s]",
        "Takeoff Peak Force (Right) [N]",
        "Takeoff Peak Force (Left) [N]",
        "Concentric Time to Peak Force (Left) [ms]",
        "Concentric Time to Peak Force (Right) [ms]",
        "Force at Peak Power (Left) [N]",
        "Force at Peak Power (Right) [N]",
        "Peak Landing Force (Left) [N]",
        "Peak Landing Force (Right) [N]",
        "Landing RFD (Right) [N/s]",
        "Landing RFD (Left) [N/s]",
        "Concentric Peak Force (Left) [N]",
        "Concentric Peak Force (Right) [N]",
        "Eccentric Peak Force (Left) [N]",
        "Eccentric Peak Force (Right) [N]",
    ]
    cols_mostrar = [c for c in columnas_base if c in df_tmp.columns]

    while True:
        conteo = df_tmp["Athlete"].value_counts().reset_index()
        conteo.columns = ["Athlete", "Trials"]
        reprobados = conteo[conteo["Trials"] != 3]

        if reprobados.empty:
            print(f"No hay atletas con un número de intentos distinto de 3 "
                  f"en la sesión '{sheet_name}'.")
            return df_tmp

        df_reprobados = df_tmp.loc[
            df_tmp["Athlete"].isin(reprobados["Athlete"]), cols_mostrar
        ].sort_values("Athlete")

        print(f"\n=== Sesión '{sheet_name}' ===")
        print("Atletas con intentos distintos de 3:")
        display(reprobados)
        print("Sus saltos (el número de la izquierda es el que escribes):")
        display(df_reprobados)

        while True:
            accion = input(
                "\n¿Qué quieres hacer?\n"
                "  [1] Eliminar a todos los atletas de la lista\n"
                "  [2] Revisar atletas específicos\n"
                "  [3] Terminar sin eliminar más\n"
                "Opción: "
            ).strip()
            if accion in ("1", "2", "3"):
                break
            print("Introduce 1, 2 o 3.")

        if accion == "1":
            df_tmp = df_tmp.loc[
                ~df_tmp["Athlete"].isin(reprobados["Athlete"])
            ].copy()
            print("Se eliminaron los atletas de la lista.")
            return df_tmp

        if accion == "3":
            return df_tmp

        nombres = pedir_nombres(set(reprobados["Athlete"]))
        if nombres is None:
            continue

        saltos = df_tmp.loc[df_tmp["Athlete"].isin(nombres), cols_mostrar]
        print("\nSaltos de los atletas seleccionados:")
        display(saltos)

        numeros = pedir_saltos(set(saltos.index))
        if numeros is None:
            continue

        print("\nSe eliminarán estos saltos:")
        display(df_tmp.loc[numeros, cols_mostrar])

        if preguntar_sn(f"¿Eliminar {len(numeros)} salto(s)? (s/n): "):
            df_tmp = df_tmp.drop(index=numeros)
            print("Saltos eliminados.")
        else:
            print("No se eliminó nada.")

        if not preguntar_sn("¿Seguir revisando? (s/n): "):
            return df_tmp

### Main


In [ ]:
def main():
    count_number(target_folder)
    sports, files = ramas_d()

    sport_input = input("Ingrese el numero del deporte a procesar: ")

    if not (sport_input.isdigit() and 1 <= int(sport_input) <= len(files)):
        print("Deporte no encontrado.")
        return {}

    file_path = os.path.join(target_folder, files[int(sport_input) - 1])
    xl = pd.ExcelFile(file_path)
    print(f"Sesiones: {xl.sheet_names}")

    sesiones_finales = {}

    for sheet_name in xl.sheet_names:
        try:
            df = pd.read_excel(file_path, sheet_name=sheet_name, header=8)
            sesiones_finales[sheet_name] = jump_count(df, sheet_name)

        except Exception as e:
            print(f"  Error al procesar la hoja '{sheet_name}': {e}")
            traceback.print_exc()

    return sesiones_finales


sesiones_finales = main()

In [ ]:
show(**sesiones_finales)                                # todas las sesiones en PandasGUI
todo = pd.concat(sesiones_finales, names=["Sesion"])    # un solo DataFrame para analizar